In [ ]:
%pip install gymnasium

In [ ]:
%pip install "tqdm[notebook]"

In [ ]:
%pip install ImageIO

In [1]:
%pip install "gymnasium[toy-text]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 22.7 MB/s  0:00:00m0:00:010:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import gymnasium as gym
import random
import imageio
from tqdm.notebook import trange

In [3]:
# 1. Configuración del Entorno
# En gymnasium, se recomienda especificar el render_mode al crear el entorno
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
    )

# 2. Inicialización de la Q-table
state_space = env.observation_space.n
action_space = env.action_space.n

def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))

Qtable_frozenlake = initialize_q_table(state_space, action_space)

# 3. Políticas
def epsilon_greedy_policy(Qtable, state, epsilon):
    random_int = random.uniform(0, 1)
    if random_int > epsilon:
        action = np.argmax(Qtable[state])
    else:
        action = env.action_space.sample()
    return action

def greedy_policy(Qtable, state):
    return np.argmax(Qtable[state])

# 4. Hiperparámetros
n_training_episodes = 10000
learning_rate = 0.7
n_eval_episodes = 100
max_steps = 99
gamma = 0.95
max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.0005

# 5. Entrenamiento
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
    for episode in trange(n_training_episodes):
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        
        # CAMBIO: gymnasium.reset() devuelve (state, info)
        state, info = env.reset()
        done = False

        for step in range(max_steps):
            action = epsilon_greedy_policy(Qtable, state, epsilon)

            # CAMBIO: gymnasium.step() devuelve 5 valores (truncado y terminado por separado)
            new_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            # Actualización de Q-table
            Qtable[state][action] = Qtable[state][action] + learning_rate * (
                reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action]
            )

            if done:
                break
            state = new_state
    return Qtable

Qtable_frozenlake = train(
    n_training_episodes,
    min_epsilon,
    max_epsilon,
    decay_rate,
    env,
    max_steps,
    Qtable_frozenlake
    )

# 6. Evaluación
def evaluate_agent(env, max_steps, n_eval_episodes, Q):
    episode_rewards = []
    for episode in range(n_eval_episodes):
        state, info = env.reset()
        total_rewards_ep = 0

        for step in range(max_steps):
            action = np.argmax(Q[state][:])
            new_state, reward, terminated, truncated, info = env.step(action)
            total_rewards_ep += reward

            if terminated or truncated:
                break
            state = new_state
        episode_rewards.append(total_rewards_ep)
    
    mean_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)
    return mean_reward, std_reward

mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake)
print(f"Resultado de la evaluación: Media={mean_reward:.2f} +/- {std_reward:.2f}")

# 7. Grabación de Video
def record_video(env, Qtable, out_directory, fps=1):
    images = []
    state, info = env.reset()
    # CAMBIO: El renderizado depende del render_mode definido en gym.make
    img = env.render()
    images.append(img)
    
    done = False
    while not done:
        action = np.argmax(Qtable[state][:])
        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        img = env.render()
        images.append(img)
    
    imageio.mimsave(out_directory, [np.array(img) for img in images], fps=fps)

video_path = "replay.gif"
record_video(env, Qtable_frozenlake, video_path, fps=1)

  0%|          | 0/10000 [00:00<?, ?it/s]

Resultado de la evaluación: Media=1.00 +/- 0.00
